In [1]:
# Some basic setup:
# Setup detectron2 logger
import detectron2
from detectron2.utils.logger import setup_logger
setup_logger()

# import some common libraries
import numpy as np
import os, json, cv2, random
import matplotlib.pyplot as plt
%matplotlib inline

# import some common detectron2 utilities
from detectron2 import model_zoo
from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg
from detectron2.utils.visualizer import Visualizer
from detectron2.data import MetadataCatalog, DatasetCatalog
from detectron2.utils.visualizer import ColorMode

from train import register_dataset

/home/abhinavchadaga/cs/fri_II/final_project/.venv/lib/python3.8/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
path_to_dataset = "datasets/elevator_panels"

In [3]:
DatasetCatalog.register("elevators", lambda: register_dataset(path_to_dataset))
MetadataCatalog.get("elevators").thing_classes = ["button", "label", "not button"]

In [4]:
cascade_mask_rcnn_config = "Misc/cascade_mask_rcnn_R_50_FPN_3x.yaml"
mask_rcnn_config = "COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"

cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file(cascade_mask_rcnn_config))
cfg.DATASETS.TRAIN = ("elevators", )
cfg.DATASETS.TEST = ()
cfg.MODEL.WEIGHTS = "/home/abhinavchadaga/cs/fri_II/final_project/output/model_final.pth"
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 3

cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5
predictor = DefaultPredictor(cfg)

[12/02 10:56:41 d2.checkpoint.c2_model_loading]: Following weights matched with model:
| Names in Model                                  | Names in Checkpoint                                                                                  | Shapes                                          |
|:------------------------------------------------|:-----------------------------------------------------------------------------------------------------|:------------------------------------------------|
| backbone.bottom_up.res2.0.conv1.*               | backbone.bottom_up.res2.0.conv1.{norm.bias,norm.running_mean,norm.running_var,norm.weight,weight}    | (64,) (64,) (64,) (64,) (64,64,1,1)             |
| backbone.bottom_up.res2.0.conv2.*               | backbone.bottom_up.res2.0.conv2.{norm.bias,norm.running_mean,norm.running_var,norm.weight,weight}    | (64,) (64,) (64,) (64,) (64,64,3,3)             |
| backbone.bottom_up.res2.0.conv3.*               | backbone.bottom_up.res2.0.conv3.{norm.bia

In [6]:
im = cv2.imread("/home/abhinavchadaga/cs/fri_II/final_project/instance_segmentation/waterloo.jpg")
outputs = predictor(im)

/home/abhinavchadaga/cs/fri_II/final_project/.venv/lib/python3.8/site-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3190.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


In [8]:
v = Visualizer(im[:,:,::-1], MetadataCatalog.get("elevators"), scale=1, instance_mode=ColorMode.IMAGE)
out = v.draw_instance_predictions(outputs["instances"].to("cpu"))
cv2.imwrite("output.jpg", out.get_image()[:, :, ::-1])

True

In [ ]:
import numpy as np

print(outputs["instances"].pred_masks.shape)
for i in range(outputs["instances"].pred_masks.shape[0]):
    mask = outputs["instances"].pred_masks[i, :, :].to("cpu").numpy().astype(np.uint8)
    plt.imshow(mask, cmap="gray")
    plt.show()